# AFM Data Analysis: Starter Template

A working starting point for analysing your own AFM data with the tools from the
workshop. Run each cell top to bottom, editing the marked parameters to suit your
sample.

**How to use this notebook**
- Look for `# --- EDIT THIS ---` to find every place that needs your input.
- Read the short comment above each block: it says what that step does and when
  you might want to change it.
- If something looks wrong, check the image at the end of each section before
  continuing; it's easier to spot problems early.

**Contents**
1. Setup
2. Load your data
3. Inspect the raw image
4. Level the data
5. Analyse: roughness *or* feature measurement
6. Export

---

## 1 · Setup

Run this cell once at the start. It imports everything the notebook needs and
registers the workshop's AFM colormaps.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize

from scipy.ndimage import gaussian_filter
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops_table
from skimage.segmentation import clear_border
from skimage.color import label2rgb
import tifffile

from AFMReader.general_loader import LoadFile

# --- Colormap registration (same as the workshop notebooks) ---
def register_afm_colormaps(folder=Path(".") / "resources" / "colormaps"):
    for name in ("afm_brown", "playnano_gold", "classic_afm", "aurea"):
        if name in plt.colormaps():
            continue
        try:
            rgb_data = np.loadtxt(folder / f"{name}.csv", delimiter=",")
            cmap = ListedColormap(rgb_data, name=name)
            plt.colormaps.register(cmap)
            plt.colormaps.register(cmap.reversed(), name=f"{name}_r")
        except FileNotFoundError:
            pass

register_afm_colormaps()
print("Ready.")

## Shared functions

These are the same functions built in the workshop notebooks — no changes needed here.
Run this cell before any of the analysis sections below.

In [ ]:
# --- From notebook 02 ---

def fit_plane(data, mask=None):
    """Return the best-fit plane. Fit background only if mask given (True=feature)."""
    h, w = data.shape
    Y, X = np.mgrid[0:h, 0:w]
    coords = np.column_stack([X.ravel(), Y.ravel()])
    z = data.ravel()
    keep = np.ones(z.shape, bool) if mask is None else ~mask.ravel()
    model = LinearRegression().fit(coords[keep], z[keep])
    return model.predict(coords).reshape(h, w)

def fit_polynomial(data, order=2, mask=None):
    """Return best-fit 2D polynomial surface (background only if masked)."""
    h, w = data.shape
    Y, X = np.mgrid[0:h, 0:w]
    coords = np.column_stack([X.ravel(), Y.ravel()])
    z = data.ravel()
    A = PolynomialFeatures(order).fit_transform(coords)
    keep = np.ones(z.shape, bool) if mask is None else ~mask.ravel()
    coeff, *_ = np.linalg.lstsq(A[keep], z[keep], rcond=None)
    return (A @ coeff).reshape(h, w)

def row_median_align(data, mask=None):
    """Subtract each row's median (background pixels only, if mask given)."""
    out = data.copy()
    for i in range(data.shape[0]):
        ref = data[i, :] if mask is None else data[i, ~mask[i, :]]
        if ref.size == 0:
            ref = data[i, :]
        out[i, :] -= np.median(ref)
    return out

def threshold_mask(data, k=1.0):
    """Boolean mask: True where height > mean + k*std (True = feature)."""
    return data > (data.mean() + k * data.std())

def level_plane_lines(img, k=1.0, line_order=1):
    """Plane + masked per-row polynomial. Best for sparse features + line offsets."""
    rough = row_median_align(img - fit_plane(img))
    mask  = threshold_mask(rough, k=k)
    out   = img - fit_plane(img, mask=mask)
    out   = row_polynomial_align(out, order=line_order, mask=mask)
    return out, mask

def row_polynomial_align(data, order=1, mask=None):
    """Fit and subtract a polynomial along each row (background only if masked)."""
    out = data.astype(np.float64).copy()
    x   = np.arange(data.shape[1])
    for i in range(data.shape[0]):
        keep = np.ones(x.shape, bool) if mask is None else ~mask[i, :]
        if keep.sum() <= order:
            out[i, :] -= np.median(data[i, keep] if keep.any() else data[i, :])
            continue
        coeffs = np.polyfit(x[keep], data[i, keep], order)
        out[i, :] -= np.polyval(coeffs, x)
    return out

def level_polynomial(img, k=1.0, order=2):
    """Plane + masked 2D polynomial + masked row alignment. Best for bow."""
    rough = row_median_align(img - fit_plane(img))
    mask  = threshold_mask(rough, k=k)
    out   = img - fit_polynomial(img, order=order, mask=mask)
    out   = row_median_align(out, mask=mask)
    return out, mask

# --- From notebook 03 ---

def Sq(data, mask=None):
    """Root-mean-square roughness. Excludes masked (True) pixels if given."""
    vals = data if mask is None else data[~mask]
    return np.sqrt(np.mean((vals - vals.mean())**2))

def Sa(data, mask=None):
    """Mean absolute roughness. Excludes masked (True) pixels if given."""
    vals = data if mask is None else data[~mask]
    return np.mean(np.abs(vals - vals.mean()))

# --- From notebook 04 ---

def show(img, title, cmap="afm_brown", zmin=None, zmax=None, ax=None, colorbar=True):
    ax = ax or plt.gca()
    im = ax.imshow(img, cmap=cmap, vmin=zmin, vmax=zmax, origin="lower")
    ax.set_title(title); ax.axis("off")
    if colorbar:
        plt.colorbar(im, ax=ax, label="Height (nm)", fraction=0.046)
    return im

def nice_round_length(target):
    """Round to the nearest 1, 2, or 5 x a power of ten."""
    if target <= 0:
        return 1
    exponent = np.floor(np.log10(target))
    fraction = target / 10**exponent
    if fraction < np.sqrt(2):   nice = 1
    elif fraction < np.sqrt(10): nice = 2
    elif fraction < np.sqrt(50): nice = 5
    else:                        nice = 10
    return nice * 10**exponent

def draw_scalebar(ax, pixel_to_nm, image_width_px, fraction=0.2,
                  color="white", margin_frac=0.04, units="nm"):
    """Draw an auto-length scale bar. units: 'nm' or 'um'."""
    target_nm = fraction * image_width_px * pixel_to_nm
    bar_nm = nice_round_length(target_nm)
    bar_px = bar_nm / pixel_to_nm
    bar_value = bar_nm / 1000 if units == "um" else bar_nm
    unit_label = "µm" if units == "um" else "nm"
    margin = margin_frac * image_width_px
    x0 = image_width_px - margin - bar_px
    y0 = margin
    height_px = 0.015 * image_width_px
    ax.add_patch(plt.Rectangle((x0, y0), bar_px, height_px, color=color))
    ax.text(x0 + bar_px / 2, y0 + height_px * 3, f"{bar_value:g} {unit_label}",
            color=color, ha="center", fontsize=11)

print("Functions ready.")

## 2 · Load your data

Point this at your file and pick the right channel name. If you're unsure of the
channel name, try loading it and AFMReader's error message will list the available
ones.

In [ ]:
# --- EDIT THIS: path to your data file ---
# Use a raw string (r"...") if your path has backslashes (Windows)
DATA_PATH = Path(r"PATH/TO/YOUR/FILE.spm")

# --- EDIT THIS: channel name for your file format ---
# Common values:
#   Bruker .spm       -> "Height" or "Height Sensor" or "ZSensor"
#   JPK .jpk          -> "height_retrace" or "height_trace"
#   Asylum .ibw       -> "HeightRetrace"
#   HS-AFM .h5-jpk   -> "height_trace"
CHANNEL = "Height"

img, pixel_to_nm = LoadFile(DATA_PATH, channel=CHANNEL).load()
img = img.astype(np.float64)

print(f"Loaded:     {DATA_PATH.name}")
print(f"Shape:      {img.shape}  (height x width, in pixels)")
print(f"Pixel size: {pixel_to_nm:.4f} nm/pixel")
print(f"Field of view: {img.shape[1] * pixel_to_nm / 1000:.2f} x {img.shape[0] * pixel_to_nm / 1000:.2f} µm")

## 3 · Inspect the raw image

Look at the raw data before doing anything to it. Things to notice:

- Is there an obvious **tilt** (the image is darker on one side than the other)?
- Are there visible **horizontal stripes** (line offsets)?
- Does the background look gently **curved** (bow)?

The line profile helps make tilt and bow visible; a flat background should be roughly horizontal.

In [ ]:
# --- EDIT THIS: adjust zmin/zmax if the colour scale looks washed out ---
# Leave as None to use the data's own range automatically.
ZMIN = None
ZMAX = None

# --- EDIT THIS: colormap ---
# Options: "afm_brown", "playnano_gold", "aurea", "classic_afm", "gray", "afmhot", "viridis" etc.
CMAP = "afm_brown"

row = img.shape[0] // 2
x_nm = np.arange(img.shape[1]) * pixel_to_nm

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
show(img, "Raw image", cmap=CMAP, zmin=ZMIN, zmax=ZMAX, ax=ax[0])
ax[0].axhline(row, color="r", lw=1, ls="--")
ax[1].plot(x_nm, img[row, :])
ax[1].set_xlabel("Distance (nm)"); ax[1].set_ylabel("Height (nm)")
ax[1].set_title(f"Row profile (row {row}) — look for tilt or bow")
plt.tight_layout(); plt.show()

## 4 · Level the data

**Choose the method that matches what you saw in Section 3.**

Run *one* of the three cells below, then skip to Section 4.3 to check the result.

### 4.1 · Plane fit only

Use this when: tilt is the only visible artifact, the background looks roughly flat
after correcting for the slope, and there are no horizontal stripes.

In [ ]:
# --- EDIT THIS: k controls how aggressively features are masked before re-levelling.
#   Higher k = tighter mask (only very tall features excluded).
#   Lower k  = broader mask (more pixels excluded, better for dense samples).
#   Start at 1.0 and adjust if the levelled image looks over- or under-corrected.
k = 1.0

rough = img - fit_plane(img)
mask  = threshold_mask(rough, k=k)
levelled = img - fit_plane(img, mask=mask)

print("Plane fit done.")
print(f"Mask covers {100*mask.mean():.1f}% of the image (higher k -> smaller mask).")

### 4.2a · Plane + row alignment

Use this when: there are **visible horizontal stripes** as well as tilt.
Applies a masked per-row polynomial after the plane fit.

In [ ]:
# --- EDIT THIS ---
k = 1.0            # mask threshold (see notes in 4.1)
line_order = 1     # 1 = remove per-row tilt; 2 = also remove per-row curvature

levelled, mask = level_plane_lines(img, k=k, line_order=line_order)

print("Plane + row alignment done.")
print(f"Mask covers {100*mask.mean():.1f}% of the image.")

### 4.2b · Polynomial fit (for bow)

Use this when: the background curves gently across the whole image (scanner bow) and a
plane alone leaves a residual curve in the line profile.

In [ ]:
# --- EDIT THIS ---
k = 1.0     # mask threshold (see notes in 4.1)
order = 2   # polynomial order: 2 is usually sufficient for typical scanner bow

levelled, mask = level_polynomial(img, k=k, order=order)

print("Polynomial fit done.")
print(f"Mask covers {100*mask.mean():.1f}% of the image.")

### 4.3 · Check the result

Run this after whichever levelling cell you chose above.

In [ ]:
row = levelled.shape[0] // 2
x_nm = np.arange(levelled.shape[1]) * pixel_to_nm

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
show(levelled, "Levelled image", cmap=CMAP, ax=ax[0])
ax[0].axhline(row, color="r", lw=1, ls="--")
ax[1].plot(x_nm, levelled[row, :])
ax[1].axhline(0, color="k", lw=0.5)
ax[1].set_xlabel("Distance (nm)"); ax[1].set_ylabel("Height (nm)")
ax[1].set_title("Row profile after levelling — background should be near zero")
plt.tight_layout(); plt.show()

bg = ~mask
print(f"Background mean (should be near 0): {levelled[bg].mean():+.3f} nm")

If none of these options well level the data try constructing your own workflow from the funcitons above or by writing your own.

## 5 · Analyse

**Choose the analysis that suits your sample.**

- **Section 5.1**: surface roughness (Sq, Sa): use this for films, substrates, or
  any sample where you want a single number describing how rough the surface is.
- **Section 5.2**: feature measurement (size, shape, height): use this for discrete
  objects (molecules, particles, grains) that you want to segment and measure
  individually.

You can run both if your sample has both a rough background and discrete features.

### 5.1 · Surface roughness (Sq and Sa)

In [ ]:
# --- EDIT THIS: k controls which pixels are treated as surface 'blobs' to exclude.
#   Try k=2.5 to mask only clear high features; lower k masks more.
#   Set use_mask=False to compute roughness over the whole image without masking.
k_roughness = 2.5
use_mask = True

roughness_mask = threshold_mask(levelled, k=k_roughness) if use_mask else None

print(f"Sq (RMS roughness): {Sq(levelled, mask=roughness_mask):.3f} nm")
print(f"Sa (mean roughness): {Sa(levelled, mask=roughness_mask):.3f} nm")
if use_mask:
    print(f"({100*roughness_mask.mean():.1f}% of image masked as features)")
    print()
    print("Compare with unmasked values:")
    print(f"  Sq (unmasked): {Sq(levelled):.3f} nm")
    print(f"  Sa (unmasked): {Sa(levelled):.3f} nm")

### 5.2 · Feature measurement

Segments individual features and measures their size, shape, and height.

In [ ]:
# --- EDIT THIS: k controls the threshold used to detect features.
#   Lower k catches fainter features; higher k is more selective.
k_analysis = 1.5
analysis_mask = threshold_mask(levelled, k=k_analysis)

# Label connected regions
labeled = label(analysis_mask)

# Remove anything touching the frame edge — clipped features give wrong measurements
labeled = clear_border(labeled)
n_found = len(np.unique(labeled)) - 1   # -1 for background
print(f"Found {n_found} regions after removing edge-touching objects.")

# Show the labelled result
plt.imshow(labeled, cmap="nipy_spectral", origin="lower")
plt.title(f"{n_found} regions found"); plt.axis("off")
plt.colorbar(label="label ID", fraction=0.046)
plt.show()

In [ ]:
# Measure shape properties
props = regionprops_table(
    labeled,
    properties=("label", "area", "axis_major_length", "axis_minor_length",
                "eccentricity", "feret_diameter_max", "orientation", "centroid"),
)
df = pd.DataFrame(props)

# Measure per-feature height by indexing the levelled image
mean_h, median_h, max_h = [], [], []
for lbl in df["label"]:
    px = levelled[labeled == lbl]
    mean_h.append(px.mean())
    median_h.append(np.median(px))
    max_h.append(px.max())
df["height_mean_nm"]   = mean_h
df["height_median_nm"] = median_h
df["height_max_nm"]    = max_h

# --- EDIT THIS: minimum area in pixels to count as a real feature (not noise).
#   Rough guide: if your features are ~X nm wide, min area ≈ (X / pixel_to_nm)²
min_area_px = 50

n_before = len(df)
df = df[df["area"] >= min_area_px].reset_index(drop=True)
print(f"Kept {len(df)} of {n_before} regions after area filtering (>= {min_area_px} px).")

# Convert lengths to nm, areas to nm²
df["major_nm"]     = df["axis_major_length"] * pixel_to_nm
df["minor_nm"]     = df["axis_minor_length"] * pixel_to_nm
df["feret_nm"]     = df["feret_diameter_max"] * pixel_to_nm
df["area_nm2"]     = df["area"] * pixel_to_nm**2
df["aspect_ratio"] = df["major_nm"] / df["minor_nm"]

df[["label", "major_nm", "minor_nm", "feret_nm", "aspect_ratio",
    "eccentricity", "height_mean_nm", "height_max_nm"]].round(2)

In [ ]:
# Coloured overlay of kept regions on the levelled image
kept_labels = set(df["label"])
filtered_labels = np.where(np.isin(labeled, list(kept_labels)), labeled, 0)

# --- EDIT THIS: zmin/zmax for the background colour scale ---
zmin_overlay = levelled.min()
zmax_overlay = levelled.max()

norm = Normalize(vmin=zmin_overlay, vmax=zmax_overlay)
img_rgb = plt.get_cmap("afm_brown")(norm(levelled))[..., :3]
overlay = label2rgb(filtered_labels, image=img_rgb, bg_label=0, alpha=0.4, saturation=1)

plt.figure(figsize=(6, 6))
plt.imshow(overlay, origin="lower")
plt.title(f"{len(df)} regions kept (coloured overlay)")
plt.axis("off"); plt.show()

# Summary statistics
summary = df[["major_nm", "minor_nm", "feret_nm", "aspect_ratio",
              "eccentricity", "height_mean_nm", "height_max_nm"]].agg(["mean", "std"])
print(summary.round(3))

## 6 · Export

### 6.1 · Save a publication-ready figure

In [ ]:
# --- EDIT THIS ---
OUTPUT_DIR = Path(".") / "data" / "figures"
FIGURE_NAME = "my_figure"     # filename, without extension

# --- EDIT THIS: scale bar units ---
# "nm" for nanometre images; "um" if your field of view is several microns
SCALEBAR_UNITS = "nm"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(5, 5))
show(levelled, "", cmap=CMAP, zmin=ZMIN, zmax=ZMAX, ax=ax)
draw_scalebar(ax, pixel_to_nm=pixel_to_nm,
              image_width_px=levelled.shape[1], units=SCALEBAR_UNITS)

# --- EDIT THIS: format and DPI ---
# PNG: good for screen/slides; PDF/SVG: vector, best for publications
# 300 DPI is the typical journal minimum
plt.savefig(OUTPUT_DIR / f"{FIGURE_NAME}.png", dpi=300, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / f"{FIGURE_NAME}.pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR / FIGURE_NAME}.png / .pdf")

### 6.2 · Save the real height data (for ImageJ, Gwyddion, or a colleague)

This saves the actual nanometre values as a float TIFF — not a colour picture of
them. Anyone with ImageJ or tifffile can reload and reprocess the real data.

In [ ]:
# --- EDIT THIS ---
DATA_NAME = "my_levelled_data"   # filename, without extension

import tifffile
tifffile.imwrite(OUTPUT_DIR / f"{DATA_NAME}.tif", levelled.astype(np.float32))
print(f"Saved: {OUTPUT_DIR / DATA_NAME}.tif")

# Quick round-trip check
reloaded = tifffile.imread(OUTPUT_DIR / f"{DATA_NAME}.tif")
print(f"Round-trip verified: {np.array_equal(levelled.astype(np.float32), reloaded)}")
print(f"A real height value: {levelled[50,50]:.4f} nm  ->  reloaded as: {reloaded[50,50]:.4f} nm")

### 6.3 · Save the measurement table (if you ran Section 5.2)

In [ ]:
# --- EDIT THIS ---
TABLE_NAME = "my_measurements"   # filename, without extension

RESULTS_DIR = Path(".") / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(RESULTS_DIR / f"{TABLE_NAME}.csv", index=False)
print(f"Saved: {RESULTS_DIR / TABLE_NAME}.csv  ({len(df)} rows)")

---

### Quick reference

| Artifact | Recommended method |
|---|---|
| Tilt only | Section 4.1 (plane fit) |
| Tilt + horizontal stripes | Section 4.2a (plane + row alignment) |
| Background curves across image | Section 4.2b (polynomial fit) |

| Analysis | Section |
|---|---|
| Surface roughness (film, substrate) | 5.1 |
| Discrete objects (molecules, grains) | 5.2 |

**If something looks wrong:**
- Bad levelling → go back to Section 4 and try a different method or adjust `k`.
- Too many / too few features → adjust `k_analysis` or `min_area_px` in Section 5.2.
- Colour scale washed out → set `ZMIN` / `ZMAX` manually in Section 3.